# 26 — Material Fine-Tune (final booster)

Trains a focused LoRA adapter on the curated dataset under
`Train/Material/`. This is the **second** time Material data
touches the model — `03_material_seeding` already folded the
same data into `pairs.jsonl` at the start of the pipeline, so it
flowed through every upstream stage (warm-start, full SFT,
preference SFT, iterative loop, distillation). This notebook is a
**targeted boost** at the very end: a small adapter that hammers
the model on this exact dataset one more time to sharpen the
behaviour the gold examples teach.

Inputs:

* `Train/Material/curated.jsonl` — 198+ hand-curated
  `(instruction, output)` pairs that each pass `aro check`. Written
  by `Train/tools/curate_material.py`.
* `Train/Material/<slug>.json` — per-prompt failures from
  `Train/tools/run_prompts.py`. Each entry has the prompt the
  user typed, what the model actually said (the `actual` field) and
  what it should have said (the `expected` field, from
  `canonical.json`). Only pairs with a non-null `expected` are used.
* `Train/Material/correct.log` — model outputs that the runner judged
  correct. Optionally folded in as additional positive examples.

Outputs:

* `models/material/dataset/train.jsonl` + `valid.jsonl` — the assembled
  messages-format training set.
* `models/material/adapter/` — LoRA adapter written by mlx_lm.
* `models/material/fused/` — adapter fused back into the base model,
  for downstream packaging (NB24).

In [ ]:
import subprocess, sys
from pathlib import Path
_cfg = Path('.').resolve()
if str(_cfg) not in sys.path:
    sys.path.insert(0, str(_cfg))
import config as _config; import importlib; importlib.reload(_config); from config import *

import json, random, re, shutil, tempfile
from collections import Counter

ensure_mlx_lm()

MATERIAL_DIR = Path(SCRIPT_DIR).parent / 'Material'
CURATED_FILE = MATERIAL_DIR / 'curated.jsonl'
CORRECT_LOG  = MATERIAL_DIR / 'correct.log'

OUT_DIR     = MODELS_DIR / 'material'
DATA_DIR    = OUT_DIR / 'dataset'
ADAPTER_DIR = OUT_DIR / 'adapter'
FUSED_DIR   = OUT_DIR / 'fused'
# Anchor the booster on the BEST upstream model — distilled
# student preferred, then DPO-tuned teacher, then base. This
# makes NB12 a real booster instead of parallel training.
BASE_FOR_MATERIAL = str(MODEL_ID)  # fallback
_candidates = [
    MODELS_DIR / 'distill' / 'student' / 'fused',
    MODELS_DIR / 'dpo' / 'fused',
    MODELS_DIR / 'finetune' / 'round_0' / 'fused',
]
for _cand in _candidates:
    if (_cand / 'config.json').exists():
        BASE_FOR_MATERIAL = str(_cand)
        print(f'NB12 base: {_cand.name} ({_cand})')
        break
else:
    print(f'NB12 base: {MODEL_ID} (no upstream fine-tune found)')

for d in (DATA_DIR, ADAPTER_DIR, FUSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Material dir: {MATERIAL_DIR}')
print(f'  curated.jsonl  → {CURATED_FILE.exists()}')
print(f'  per-prompt jsons → {len(list(MATERIAL_DIR.glob("*.json"))) - 1} (excluding canonical.json)')

## Collect pairs

In [ ]:
kb = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)

def to_messages(instruction: str, output: str) -> dict:
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': instruction},
            {'role': 'assistant', 'content': output},
        ],
        'task_type': 'code_generation',
    }

pairs = []
source_counts = Counter()

# 1. Hand-curated examples (always present, all validated).
if CURATED_FILE.exists():
    with open(CURATED_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            ex = json.loads(line)
            pairs.append(to_messages(ex['instruction'], ex['output']))
            source_counts['curated'] += 1

# 2. Per-prompt failures from run_prompts.py — only those with a
#    canonical answer in canonical.json. Skip the canonical file itself.
for p in sorted(MATERIAL_DIR.glob('*.json')):
    if p.name == 'canonical.json':
        continue
    try:
        rec = json.loads(p.read_text())
    except json.JSONDecodeError:
        continue
    expected = rec.get('expected')
    if not expected:
        continue
    pairs.append(to_messages(rec.get('prompt', ''), expected))
    source_counts['runner_failures'] += 1

# 3. correct.log entries — the model already nailed these, but they
#    serve as positive reinforcement. Use the prompt + a canonical
#    answer ONLY if we can find one in canonical.json; otherwise skip
#    (we don't have ground truth to train on).
canonical = {}
if (MATERIAL_DIR / 'canonical.json').exists():
    cdata = json.loads((MATERIAL_DIR / 'canonical.json').read_text())
    canonical = {k: v for k, v in cdata.items() if not k.startswith('_')}

if CORRECT_LOG.exists():
    with open(CORRECT_LOG) as f:
        for line in f:
            if not line.strip():
                continue
            entry = json.loads(line)
            prompt = entry.get('prompt', '')
            expected = canonical.get(prompt)
            if expected:
                pairs.append(to_messages(prompt, expected))
                source_counts['correct_with_canonical'] += 1

print(f'Collected {len(pairs)} training pairs:')
for src, n in source_counts.most_common():
    print(f'  {src:24} {n}')

if not pairs:
    raise SystemExit('No training pairs — run curate.py and/or run_prompts.py first.')

## Defensive `aro check` gate

Every code block in the dataset is re-validated here. The curated set is
already pre-validated by `curate.py`, but per-prompt expected answers
haven't been — fail-loud if anything got past.

In [ ]:
_ARO_FENCE = re.compile(r'```aro\n(.*?)```', re.DOTALL)

def extract_aro(text: str) -> str:
    blocks = _ARO_FENCE.findall(text or '')
    return '\n\n'.join(b.strip() for b in blocks) if blocks else ''

def aro_check_ok(code: str) -> bool:
    if not code.strip():
        return True  # prose-only answer — nothing to validate
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp],
                               capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except FileNotFoundError:
        print('WARNING: aro binary not on PATH — skipping validation')
        return True
    except subprocess.TimeoutExpired:
        return False

kept, dropped = [], 0
for p in pairs:
    assistant = p['messages'][-1]['content']
    if aro_check_ok(extract_aro(assistant)):
        kept.append(p)
    else:
        dropped += 1

print(f'After re-validation: kept {len(kept)} / dropped {dropped}')
pairs = kept

## Train / valid split

In [ ]:
random.seed(42)
random.shuffle(pairs)
split = max(1, int(len(pairs) * 0.10))
valid_pairs = pairs[:split]
train_pairs = pairs[split:]

TRAIN_FILE = DATA_DIR / 'train.jsonl'
VALID_FILE = DATA_DIR / 'valid.jsonl'
with open(TRAIN_FILE, 'w') as f:
    for p in train_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
with open(VALID_FILE, 'w') as f:
    for p in valid_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')

print(f'train: {len(train_pairs)} → {TRAIN_FILE}')
print(f'valid: {len(valid_pairs)} → {VALID_FILE}')

## Fine-tune with mlx_lm.lora

Small LoRA on top of the base model. Iteration count scales with the
dataset size — Material/ is much smaller than the full corpus, so 200–
400 iters is the right ballpark.

In [ ]:
ITERS = max(200, min(1200, len(train_pairs) * 4))  # round-2 audit: was 400 cap
BATCH = 1
LR    = 1e-4
RANK  = 8
LAYERS = 8
MAX_SEQ = 4096

cmd = [
    sys.executable, '-m', 'mlx_lm', 'lora',
    '--model',          BASE_FOR_MATERIAL,
    '--train',
    '--data',           str(DATA_DIR),
    '--adapter-path',   str(ADAPTER_DIR),
    '--iters',          str(ITERS),
    '--batch-size',     str(BATCH),
    '--learning-rate',  str(LR),
    '--num-layers',     str(LAYERS),
    '--max-seq-length', str(MAX_SEQ),
    '--save-every',     '100',
    '--steps-per-report', '20',
    '--steps-per-eval',   '50',
    '--val-batches',      '5',
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
if result.returncode != 0:
    raise RuntimeError(f'mlx_lm lora failed (exit {result.returncode})')

## Fuse adapter into the base model

In [ ]:
fuse_cmd = [
    sys.executable, '-m', 'mlx_lm', 'fuse',
    '--model',        BASE_FOR_MATERIAL,
    '--adapter-path', str(ADAPTER_DIR),
    '--save-path',    str(FUSED_DIR),
]
print(' '.join(fuse_cmd))
subprocess.run(fuse_cmd, check=True)
print(f'Fused model → {FUSED_DIR}')

## Smoke test the fine-tuned model

In [ ]:
load_fn, generate_fn, make_sampler = ensure_mlx_lm()
model, tokenizer = load_fn(str(FUSED_DIR))

PROBES = [
    'How to sum two numbers?',
    'Write a Hello World application.',
    'Store a new user in the user-repository.',
    'Branch on an HTTP status code (200, 404, 500, otherwise).',
    'Define a user-defined action that doubles a number.',
]

for probe in PROBES:
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': probe},
    ]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = generate_fn(model, tokenizer, prompt=p, max_tokens=600, verbose=False,
                     sampler=make_sampler(temp=0.2))
    code = extract_aro(out)
    ok = aro_check_ok(code) if code else False
    print(f'\n--- {probe!r} ---')
    print(out[:600])
    print(f'  aro check: {"PASS" if ok else "FAIL"}')

In [ ]:
# ── Chart: training summary + smoke-test pass rate ───────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date as _date

plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '23_material_finetune.png'

# Re-run smoke test for the chart (5 canonical probes).
_smoke = []
for probe in PROBES:
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': probe},
    ]
    p = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = generate_fn(model, tokenizer, prompt=p, max_tokens=600,
                     verbose=False, sampler=make_sampler(temp=0.2))
    code = extract_aro(out)
    _smoke.append({
        'probe': probe,
        'pass':  bool(code and aro_check_ok(code)),
    })
_pass = sum(1 for s in _smoke if s['pass'])
_fail = len(_smoke) - _pass

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: dataset composition
_labels = list(source_counts.keys())
_counts = [source_counts[k] for k in _labels]
_colors = ['#3498db', '#2ecc71', '#9b59b6'][:len(_labels)]
ax1.bar(_labels, _counts, color=_colors, edgecolor='white', width=0.6)
for i, v in enumerate(_counts):
    ax1.text(i, v, str(v), ha='center', va='bottom', fontsize=10)
ax1.set_ylabel('pairs')
ax1.set_title(f'Material training set ({sum(_counts)} pairs)',
              fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_axisbelow(True)

# Right: smoke test
ax2.pie([_pass, _fail],
        labels=[f'PASS  ({_pass})', f'FAIL  ({_fail})'],
        colors=['#2ecc71', '#e74c3c'],
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title(f'Post-fine-tune smoke test  ({len(_smoke)} probes)',
              fontsize=12, fontweight='bold')

fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')
